In [ ]:
from tqdm import tqdm
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# 加载数据并预处理
data = pd.read_csv('../../data-hh/my/202410221653-processed.csv', dtype={'aircraft': str})
categorical_columns = ['flt_no', 'bd_type', 'aircraft', 'a', 'b', 'c', 'from', 'to']
label_encoders = {}
for col in categorical_columns:
    label_encoders[col] = LabelEncoder()
    data[col] = label_encoders[col].fit_transform(data[col].astype(str))
data.fillna(0, inplace=True)

# 分离特征和目标
features = ['flt_no', 'bd_type', 'cap', 'aircraft', 'legs', 'leg_no', 'duration', 
            'a', 'b', 'c', 'year', 'month', 'day', 'weekday', 'hour', 'minute', 
            'second', 'from', 'to']
target = 'pax'
X = data[features]
y = data[target]

# 划分数据集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 初始化模型
model = DecisionTreeRegressor(random_state=42)

# 模拟训练过程并添加进度条
batch_size = 100
trained_samples = 0
num_batches = len(X_train) // batch_size

for i in tqdm(range(0, len(X_train), batch_size), desc="Training Progress"):
    batch_X = X_train.iloc[i:i+batch_size]
    batch_y = y_train.iloc[i:i+batch_size]
    
    if i == 0:
        model.fit(batch_X, batch_y)
    else:
        model.fit(pd.concat([X_train.iloc[:i], batch_X]), pd.concat([y_train.iloc[:i], batch_y]))
    
    trained_samples += len(batch_X)

# 测试模型
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")


Training Progress:   7%|█████▋                                                                               | 506/7614 [01:34<44:51,  2.64it/s]